# Jogadores — `padel_analytics` (só o PlayerTracker)

Corre **apenas** a deteção de jogadores do João Silva (`yolov8m` + `polygon_zone` + **ByteTrack**)
sobre o `Parada4.mp4`, e descarrega o `players_detections.json` com as **boxes em píxeis**.

**Não treina nada. Não improvisa detetor nenhum.** É ligar o que já existe.

Porquê só este tracker (e não o `main.py` inteiro): o `main.py` corre 4 modelos e obriga a
clicar 12 keypoints do campo (só precisos para a homografia → metros). Nós queremos **píxeis**.
Aqui só precisas de **4 cliques** (os cantos), e corre em ~1/4 do tempo.

Antes de começar: `Runtime → Change runtime type → T4 GPU`.


## 0. Onde vai buscar o código e os pesos

Este notebook já está **independente do João Silva**:
- o **repo** vem do teu fork, `github.com/Vasco-Rocha/padel_analytics` (não do dele)
- os **pesos** `yolov8m.pt` vêm só da **tua Drive** (`MyDrive/PadelPro/pesos`), com backup local no Mac

Não há nenhum passo aqui que leia ou escreva na Drive ou no repo do João. Se um dia ele apagar
qualquer um dos dois, este notebook continua a funcionar sem alterações.


In [ ]:
# --- de onde vem o codigo ---
# fork feito em 12/07/2026: github.com/Vasco-Rocha/padel_analytics
# Independente do Joao: nao ha' aqui nenhum URL do repo ou da Drive dele.
REPO_URL = 'https://github.com/Vasco-Rocha/padel_analytics.git'

# --- onde estao os NOSSOS pesos (na tua Drive) ---
GUARDAR_NA_DRIVE = True
DRIVE_PESOS = '/content/drive/MyDrive/PadelPro/pesos'

print('repo :', REPO_URL)
print('pesos:', DRIVE_PESOS if GUARDAR_NA_DRIVE else '(so temporarios)')

## 1. GPU + clonar o repo + instalar

In [ ]:
!nvidia-smi -L
%cd /content
![ -d padel_analytics ] || git clone $REPO_URL padel_analytics
%cd /content/padel_analytics

# `parse` e `pims` sao precisos porque o ball_tracker do Joao importa-os
# (o Python carrega o modulo todo, mesmo que so' queiramos o PlayerTracker).
# versoes fixadas em 12/07/2026 (descobertas com pip freeze) -- para o Colab parar de partir
!pip install -q ultralytics==8.4.92 supervision==0.29.1 opencv-python-headless==4.13.0.92 gdown==5.2.2 parse==1.22.1 pims==0.7
print('OK')

## 2. Pesos — só da TUA Drive

In [ ]:
import os, shutil

DST = '/content/padel_analytics/weights/players_detection/yolov8m.pt'
os.makedirs(os.path.dirname(DST), exist_ok=True)

if GUARDAR_NA_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    meu = os.path.join(DRIVE_PESOS, 'yolov8m.pt')
else:
    meu = None

# So' usamos a NOSSA copia (Drive do Vasco). Nao ha' fallback para a Drive do Joao --
# se um dia ele desaparecer, isto continua a funcionar sem depender dele.
if not (meu and os.path.exists(meu)):
    raise FileNotFoundError(
        f'Nao encontrei os pesos em {meu}.\n'
        'Sobe o yolov8m.pt para a tua Drive em MyDrive/PadelPro/pesos/ '
        '(ou copia-o do backup local no Mac) antes de correr esta celula.'
    )

shutil.copy(meu, DST)
print('Pesos vindos da TUA Drive.')
print('Pesos OK:', os.path.exists(DST), f'({os.path.getsize(DST)/1e6:.0f} MB)')

## 3. Vídeo — faz upload do `Parada4.mp4`
(É o vídeo do ground-truth: 12 rallies / 117s.)

In [ ]:
from google.colab import files
import os
if not os.path.exists('/content/Parada4.mp4'):
    up = files.upload()          # escolhe o Parada4.mp4
    src = list(up.keys())[0]
    os.rename(src, '/content/Parada4.mp4')
VIDEO_PATH = '/content/Parada4.mp4'

import cv2
cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS)
N   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f'{VIDEO_PATH}: {W}x{H}, {FPS:.2f} fps, {N} frames ({N/FPS:.0f}s)')

## 4. Zona de deteção — o **frame inteiro**

Nada para clicar, nada para carregar.

O `PlayerTracker` do João pede uma `polygon_zone` para descartar quem está fora do campo.
**Damos-lhe o frame todo.** Porquê:

- A diretriz manda **nunca perder um jogador**. Um polígono mal encostado ao vidro *corta* jogadores —
  e um jogador perdido parte a formação e mata o ponto. Um espectador a mais custa uma linha de filtro.
- A limpeza fina faz-se **depois**, no repo, com a `filtrar_espectadores()`: usa a **geometria exata**
  das linhas que desenhaste (os pés nunca passam do vidro) + a regra dos **imóveis** (um jogador move-se;
  um espectador não). Corre em segundos, sem GPU, e pode iterar-se à vontade.

Custo: o detetor gasta um pouco mais de tempo com gente que vai ser deitada fora. Irrelevante.


In [ ]:
import numpy as np, supervision as sv

video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)
W, H = video_info.resolution_wh
CAMPO = np.array([[0,0],[W-1,0],[W-1,H-1],[0,H-1]], dtype=np.int32)   # frame inteiro
print(f'Zona de detecao: frame inteiro ({W}x{H}). Nao corta ninguem.')
print('A limpeza (publico, imoveis) faz-se depois, no repo, com a geometria exata do campo.')

### 4b. — (sem uso)

In [ ]:
# nada a fazer

## 5. Correr **só** o PlayerTracker do João

Importa a classe `PlayerTracker` do repo dele — **sem a reescrever**.
`yolov8m` + `polygon_zone` (o campo da tua calibração) + `ByteTrack` (IDs estáveis).

**Sem `max_det`** — deteta toda a gente, e é o polígono que descarta a bancada.
Cortar às 4 deteções *antes* de saber quem é jogador foi o bug de ontem: o espectador
ocupava uma vaga e sobrava lugar para 3 jogadores só.


In [ ]:
%cd /content/padel_analytics
import numpy as np, supervision as sv
from tqdm import tqdm
from trackers.players_tracker.players_tracker import PlayerTracker

video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)
# o `supervision` recente removeu `frame_resolution_wh`; o repo do Joao nao fixa versao
try:
    polygon_zone = sv.PolygonZone(CAMPO, frame_resolution_wh=video_info.resolution_wh)
except TypeError:
    polygon_zone = sv.PolygonZone(CAMPO)

tracker = PlayerTracker(
    'weights/players_detection/yolov8m.pt',
    polygon_zone,
    batch_size=8,
    save_path=None, load_path=None,
)
tracker.video_info_post_init(video_info)
print('device:', tracker.DEVICE, '| frames:', video_info.total_frames)
print('SEM max_det -> deteta toda a gente, o poligono e que descarta. (O bug de ontem foi cortar aos 4.)')

BATCH = 8
todos, lote = [], []
for frame in tqdm(sv.get_video_frames_generator(VIDEO_PATH), total=video_info.total_frames):
    lote.append(frame)
    if len(lote) == BATCH:
        todos.extend(tracker.predict_sample(lote)); lote = []
if lote:
    todos.extend(tracker.predict_sample(lote))

print('frames processados:', len(todos))

## 6. Gravar o JSON (boxes em **píxeis** + tracker_id) e ver a cobertura

In [ ]:
import json
serial = [p.serialize() for p in todos]
OUT = '/content/players_detections_parada4.json'
with open(OUT, 'w') as f: json.dump(serial, f)

n = len(serial) or 1
n4 = sum(1 for fr in serial if len(fr) >= 4)
n3 = sum(1 for fr in serial if len(fr) >= 3)
n1 = sum(1 for fr in serial if len(fr) >= 1)
print(f'frames            : {len(serial)}')
print(f'com os 4 jogadores: {100*n4/n:.1f}%   <-- o numero que interessa (yolov8n improvisado: 53%)')
print(f'com >=3           : {100*n3/n:.1f}%')
print(f'com >=1           : {100*n1/n:.1f}%')
print(f'media por frame   : {sum(len(fr) for fr in serial)/n:.2f}')
print('\nexemplo (frame 200):', serial[200][:1])

## 6b. ⭐ VER com os olhos — vídeo de um rally com as caixas

Antes de acreditar em qualquer número: **olha.** Isto corta um rally do ground-truth e desenha
as caixas por cima, com o **ID do ByteTrack** e o **ponto dos pés**.

O que confirmar:
1. **São 4 caixas** — e são os 4 jogadores, não público.
2. **Os IDs não trocam** quando os jogadores se cruzam ou se tapam.
3. **Os pés** (o ponto) estão no chão, dentro do campo — é o ponto que todas as regras usam.

In [ ]:
# escolhe um rally do ground-truth (t_ini, t_fim em segundos)
T_INI, T_FIM = 46.8, 67.5     # rally #2, o mais longo (20.7s)
# outros: (38.0,41.5) (77.6,85.5) (95.9,111.1) (157.9,169.4)

import cv2, numpy as np, json
from tqdm import tqdm

# --- campo por cima, em linha FINA (1px) para nao tapar as caixas ---
import os
cal = json.load(open('/content/calibracao_campo.json')) if os.path.exists('/content/calibracao_campo.json') else {}
LINHAS = [('fundo_longe_coef',(0,149,255)), ('servico_longe_coef',(255,255,255)),
          ('juncao_longe_coef',(255,0,255)), ('rede_base_coef',(150,120,255)),
          ('juncao_perto_coef',(255,170,0)), ('servico_perto_coef',(255,255,255))]
def desenhar_campo(img):
    for k, cor in LINHAS:
        if k not in cal: continue
        pts = [(x, int(np.polyval(cal[k], x))) for x in range(0, 960, 6)]
        for i in range(len(pts)-1):
            cv2.line(img, pts[i], pts[i+1], cor, 1, cv2.LINE_AA)   # 1px, suavizado
    if 'centro_coef_em_y' in cal:
        pts = [(int(np.polyval(cal['centro_coef_em_y'], y)), y) for y in range(46, 540, 6)]
        for i in range(len(pts)-1):
            cv2.line(img, pts[i], pts[i+1], (242,90,191), 1, cv2.LINE_AA)
    return img

f0, f1 = int(T_INI*video_info.fps), int(T_FIM*video_info.fps)
CORES = [(0,255,0),(0,180,255),(255,120,0),(255,0,255),(0,255,255),(255,255,0)]

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, f0)
w, h = video_info.resolution_wh
out = cv2.VideoWriter('/content/rally_boxes.mp4', cv2.VideoWriter_fourcc(*'mp4v'),
                      video_info.fps, (w, h))

n_por_frame = []
for f in tqdm(range(f0, f1)):
    ok, img = cap.read()
    if not ok: break
    img = desenhar_campo(img)
    players = todos[f] if f < len(todos) else []
    n_por_frame.append(len(players))
    for p in players:
        x1, y1, x2, y2 = [int(v) for v in p.xyxy]
        cor = CORES[(p.id or 0) % len(CORES)]
        cv2.rectangle(img, (x1,y1), (x2,y2), cor, 2)     # caixa grossa
        cv2.circle(img, p.feet, 4, cor, -1)              # PES = o que as regras usam
        cv2.putText(img, f'#{p.id}', (x1, y1-5), 0, 0.55, cor, 2)
    cv2.putText(img, f'{len(players)} jogadores', (10, 26), 0, 0.75,
                (0,255,0) if len(players)==4 else (0,0,255), 2)
    out.write(img)
cap.release(); out.release()

n = np.array(n_por_frame)
print(f'\nNESTE RALLY ({T_INI}s - {T_FIM}s, {len(n)} frames):')
for k in range(0, 7):
    c = int((n==k).sum())
    if c: print(f'   {k} jogadores: {c:4d} frames ({100*c/len(n):5.1f}%)')
print(f'\n   >>> 4 jogadores em {100*(n>=4).mean():.1f}% do rally')
# o codec mp4v nao toca em browsers -> converter para H.264
import subprocess
subprocess.run(['ffmpeg','-y','-loglevel','error','-i','/content/rally_boxes.mp4',
                '-vcodec','libx264','-pix_fmt','yuv420p','/content/rally_h264.mp4'], check=True)
print('\nvideo: /content/rally_h264.mp4  (H.264, toca no browser)')

In [ ]:
# ver no proprio Colab
from IPython.display import HTML
from base64 import b64encode
mp4 = open('/content/rally_h264.mp4','rb').read()
HTML(f'<video width=900 controls><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>')

In [ ]:
# descarregar (ver no Mac)
from google.colab import files
files.download('/content/rally_h264.mp4')

### O que fazer com o que vires

| O que vês | O que significa |
|---|---|
| **4 caixas quase sempre, IDs estáveis** | ✅ É o que precisamos. Avança para a célula 7. |
| Caixas em **público** | O polígono ficou largo (é de propósito: nunca perder um jogador). O `filtrar_espectadores()` do repo limpa isso depois, com a geometria exata do campo. |
| **Falta um jogador** com frequência | Diz-me. Vemos se é oclusão (o ByteTrack devia aguentar) ou o polígono a cortar. |
| **IDs trocam** quando se cruzam | Diz-me. Afina-se o ByteTrack. |


## 7. Descarregar
Guarda o ficheiro na pasta do projeto, em `dados_parada4/`. É este que substitui o
`player_boxes_parada4.pkl` improvisado.

In [ ]:
from google.colab import files
files.download(OUT)